# Transformer 550M — self-resuming training (English-quality edition)

Target: **~554M param** chat assistant (`target-550m` : d_model 1280, 26 layers, ctx 1024) — same size class as Qwen2.5-0.5B.

Quality English corpus: FineWeb-Edu (5 GB) + Python code (1.5 GB) + TinyStories (~1 GB).
Vocabulary: 32k BPE. Total 40,000 steps (2.6B tokens).

1. Runtime → Change runtime type → GPU (T4, free)
2. Run the cells top to bottom. Checkpoints + data live in Drive (`transformer-550m/`) and survive
   session resets — after a disconnect, just re-run cells 1-6 and training resumes from the last
   checkpoint.


In [ ]:
#@title 1 · Setup (GPU check, repo, deps)
!nvidia-smi || true
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
if not os.path.exists('transformer-llm'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/samratbarman1013-commits/transformer-llm'], check=True)
%cd transformer-llm
!pip install -q -r requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
#@title 2 · Mount Drive (checkpoint home) — Colab only
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/transformer-550m'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
print('Mounted at /content/drive, checkpoint home:', DRIVE)

In [ ]:
#@title 3 - Corpus (quality English mix ~7.5 GB - skipped if already on Drive)
import subprocess, sys, os, json, gzip, urllib.request
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)

DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)
CORPUS = f'{DATA}/corpus.txt'
NL = chr(10)

if os.path.exists(f'{DRIVE}/bins/train.bin'):
    print('corpus already prepared on Drive - nothing to download')
elif os.path.exists(CORPUS) and os.path.getsize(CORPUS) > 5_000_000_000:
    print('corpus.txt present:', round(os.path.getsize(CORPUS)/1e9, 2), 'GB')
else:
    if os.path.exists(CORPUS):
        print('incomplete corpus found (', round(os.path.getsize(CORPUS)/1e9, 2), 'GB) - redownloading')
    with open(CORPUS, 'w', encoding='utf-8') as out:
        # 1) FineWeb-Edu - high-quality educational web text (~5 GB)
        from datasets import load_dataset
        cap, n = 5_000_000_000, 0
        ds = load_dataset('HuggingFaceTB/fineweb-edu', split='train', streaming=True)
        for ex in ds:
            t = (ex.get('text') or '').strip()
            if len(t) < 200:
                continue
            out.write(t + NL); n += len(t) + 1
            if n >= cap:
                break
        print('fineweb-edu:', round(n/1e9, 2), 'GB')
        # 2) Python code (~1.5 GB) from codeparrot-clean
        cap, n = 1_500_000_000, 0
        for shard in ('file-000000000001.json.gz', 'file-000000000002.json.gz'):
            path = f'{DATA}/{shard}'
            urllib.request.urlretrieve(
                f'https://huggingface.co/datasets/codeparrot/codeparrot-clean/resolve/main/{shard}', path)
            with gzip.open(path, 'rt', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    try:
                        ex = json.loads(line)
                    except Exception:
                        continue
                    c = ex.get('content') or ''
                    if len(c) < 200 or ex.get('autogenerated'):
                        continue
                    out.write(c + NL); n += len(c) + 1
                    if n >= cap:
                        break
            os.remove(path)
            if n >= cap:
                break
        print('python code:', round(n/1e9, 2), 'GB')
        # 3) TinyStories (~1 GB slice) - simple, clean English
        ts = f'{DATA}/ts.txt'
        urllib.request.urlretrieve(
            'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt', ts)
        remaining = 1_000_000_000
        with open(ts, 'r', encoding='utf-8', errors='ignore') as f:
            while remaining > 0:
                chunk = f.read(min(10_000_000, remaining))
                if not chunk:
                    break
                out.write(chunk); remaining -= len(chunk)
        os.remove(ts)
        print('tinystories slice: 1.0 GB')
print('corpus ready')

In [ ]:
#@title 4 · Tokenizer (32k BPE) — trained once, reused across sessions
from transformer.tokenizer_bpe import train as train_tok
TOK = f'{DRIVE}/tokenizer.json'
if not (os.path.exists(TOK) and os.path.getsize(TOK) > 500_000):
    train_tok(CORPUS, TOK, vocab_size=32768)
else:
    print('reusing tokenizer from Drive')

In [ ]:
#@title 5 · Pre-tokenize to train.bin / val.bin (~30 min once; cached to Drive)
from transformer.tokenizer_bpe import encode_file
import shutil, json
BINS = f'{DATA}/bins'
DBINS = f'{DRIVE}/bins'
if not os.path.exists(f'{BINS}/train.bin'):
    if os.path.exists(f'{DBINS}/train.bin'):
        shutil.copytree(DBINS, BINS, dirs_exist_ok=True)
        print('bins restored from Drive')
    else:
        encode_file(CORPUS, TOK, BINS, val_frac=0.001)
        shutil.copytree(BINS, DBINS, dirs_exist_ok=True)
        print('bins cached to Drive — future sessions skip the download entirely')
print(json.load(open(f'{BINS}/meta.json')))

In [ ]:
#@title 6 · TRAIN (auto-resume loop — re-run this cell after any disconnect)
import time, json, shutil, os
import torch
from transformer.config import get_config
from transformer.model import GPT
from transformer.trainer import Trainer, DataLoader 

TOTAL_STEPS = 40000
CHUNK_SECONDS = 2 * 60 * 60
cfg = get_config('target-550m')
ckpt_home = f'{DRIVE}/checkpoints'

def latest_ckpt():
    ks = [k for k in os.listdir(ckpt_home) if k.startswith('ckpt_') and k.endswith('.pt')]
    return (max(ks, key=lambda k: int(k[5:-3])) if ks else None)

def run_chunk():
    cfg_out = ckpt_home + '/status.json'
    ck = latest_ckpt()
    resume = f'{ckpt_home}/{ck}' if ck else None
    m = GPT(cfg)
    if resume:
        sd = torch.load(resume, map_location='cpu')
        m.load_state_dict(sd['model'])
        print(f'resuming from {ck} (step {sd["step"]})')
    m.to('cuda')
    tr = Trainer(m, cfg, bins_dir=f'{DATA}/bins', resume=None)
    tr.fit(TOTAL_STEPS, until_seconds=CHUNK_SECONDS, resume_ckpt=resume)
    torch.cuda.empty_cache()
    
while True:
    run_chunk()
    # mirror the freshest checkpoint to Drive
    ck = latest_ckpt()
    if ck:
        src = f'{ckpt_home}/{ck}'
        shutil.copy2(src, f'{ckpt_home}/model_best.pt')
        st = {'step': None, 'updated': time.strftime('%Y-%m-%dT%H:%M:%S')}
        try:
            st['step'] = torch.load(src, map_location='cpu')['step']
        except Exception:
            pass
        json.dump(st, open(f'{ckpt_home}/status.json', 'w'))
        print('checkpoint mirrored to Drive:', ck, '-> model_best.pt')
    time.sleep(30)

In [ ]:
#@title 7 - Chat with the trained 550M model
from transformer.config import get_config
from transformer.model import GPT
from transformer.tokenizer_bpe import BpeTokenizer
import torch

tok = BpeTokenizer(f'{DRIVE}/tokenizer.json')
cfg = get_config('target-550m')
m = GPT(cfg)
sd = torch.load(f'{DRIVE}/checkpoints/model_best.pt', map_location='cpu')
m.load_state_dict(sd['model'])
m.eval().to('cuda')

def chat(prompt):
    ids = tok.encode(prompt)
    x = torch.tensor([ids], device='cuda')
    for _ in range(200):
        with torch.no_grad():
            logits = m(x)[:, -1, :]
        nxt = int(logits.argmax(-1))
        if nxt == 0:
            break
        x = torch.cat([x, torch.tensor([[nxt]], device='cuda')], 1)
    return tok.decode(x[0, len(ids):].tolist())

while True:
    p = input('you: ')
    if p.strip() in ('quit', 'exit'):
        break
    print('model:', chat(p))